## Importuri

In [1]:
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, GenerationConfig, GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, DataCollatorForLanguageModeling, Trainer
import os
import pandas as pd
import torch


## Datasets/Other variables

In [2]:
# to avoid a warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# load the dataset
ds = load_dataset("biglam/gutenberg-poetry-corpus")["train"].select(range(2000))
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.clean_up_tokenization_spaces = False
tokenizer.pad_token = tokenizer.eos_token
generator = pipeline("text-generation", model="gpt2", tokenizer=tokenizer)


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/903 [00:00<?, ?B/s]

data/train-00000-of-00001-fa9fb9e1f16eed(…):   0%|          | 0.00/91.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3085117 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Utility functions for measuring text quality

In [3]:
def lexical_diversity(text):
    words = text.lower().split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

def count_repetitions(text):
    words = text.lower().split()
    from collections import Counter
    counts = Counter(words)
    repeated = {w: c for w, c in counts.items() if c > 1}
    return repeated

def avg_word_length(text):
    words = text.split()
    if not words:
        return 0
    return sum(len(w) for w in words) / len(words)

def token_count(text, tokenizer):
    return len(tokenizer.encode(text))

def tokenize(examples):
    texts = [line + "\n" for line in examples['line']]
    result = tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

## Cerinta 1a - influenta parametrilor asupra textului generat

In [4]:

results = []
params_to_test = [
    {"temperature": 0.3, "top_p": 0.9, "repetition_penalty": 1.0},
    {"temperature": 0.7, "top_p": 0.9, "repetition_penalty": 1.0},
    {"temperature": 1.2, "top_p": 0.9, "repetition_penalty": 1.0},
    {"temperature": 0.8, "top_p": 0.3, "repetition_penalty": 1.0},
    {"temperature": 0.8, "top_p": 0.5, "repetition_penalty": 1.0},
    {"temperature": 0.8, "top_p": 0.7, "repetition_penalty": 1.0},
    {"temperature": 0.8, "top_p": 0.9, "repetition_penalty": 1.5},
    {"temperature": 0.8, "top_p": 0.9, "repetition_penalty": 2.0}
]
first_verse = "The river rose above its banks that night,"

for params in params_to_test:
    result = generator(first_verse, generation_config=GenerationConfig(
        max_new_tokens=40,
        do_sample=True,
        pad_token_id=50256,
        **params
    ))

    text = result[0]['generated_text'].replace(first_verse, "").strip()

    results.append({
        "temperature": params["temperature"],
        "top_p": params["top_p"],
        "rep_penalty": params["repetition_penalty"],
        "diversitate": round(lexical_diversity(text), 3),
        "repetitii": len(count_repetitions(text)),
        "lung_medie_cuv": round(avg_word_length(text), 2),
        "nr_tokene": token_count(text, tokenizer)
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

 temperature  top_p  rep_penalty  diversitate  repetitii  lung_medie_cuv  nr_tokene
         0.3    0.9          1.0        0.821          5            4.11         40
         0.7    0.9          1.0        0.600          7            3.86         40
         1.2    0.9          1.0        0.867          3            3.77         40
         0.8    0.3          1.0        0.361          9            4.11         40
         0.8    0.5          1.0        0.594          6            3.84         40
         0.8    0.7          1.0        0.559          7            4.18         40
         0.8    0.9          1.5        0.972          1            4.25         40
         0.8    0.9          2.0        0.971          1            3.89         40


### Influenta tokenizerului

In [5]:
test_texts = [
    "The river rose above its banks that night",
    "The flooding waters destroyed the ancient walls",
    "Dunarea a inundat orasul Melk din Austria",
    "Die Donau überschwemmte die Stadt Melk",
    "supercalifragilisticexpialidocious",
    "!!!! ??? .... ####",
]

tokenizers = {
    "GPT-2 (BPE)":        AutoTokenizer.from_pretrained("gpt2"),
    "BERT (WordPiece)":   AutoTokenizer.from_pretrained("bert-base-uncased"),
    "RoBERTa (BPE)":      AutoTokenizer.from_pretrained("roberta-base"),
    "XLM-RoBERTa (multi)":AutoTokenizer.from_pretrained("xlm-roberta-base"),
}

for text in test_texts:
    print(f"\nText: '{text}'")
    print(f"Nr cuvinte: {len(text.split())}")
    print("-" * 50)

    for nume, tok in tokenizers.items():
        tokens = tok.encode(text, add_special_tokens=False)
        token_words = tok.convert_ids_to_tokens(tokens)
        ratio = round(len(tokens) / len(text.split()), 2)

        print(f"{nume:<25} | {len(tokens):>3} tokene | ratio: {ratio} | {token_words}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Text: 'The river rose above its banks that night'
Nr cuvinte: 8
--------------------------------------------------
GPT-2 (BPE)               |   8 tokene | ratio: 1.0 | ['The', 'Ġriver', 'Ġrose', 'Ġabove', 'Ġits', 'Ġbanks', 'Ġthat', 'Ġnight']
BERT (WordPiece)          |   8 tokene | ratio: 1.0 | ['the', 'river', 'rose', 'above', 'its', 'banks', 'that', 'night']
RoBERTa (BPE)             |   8 tokene | ratio: 1.0 | ['The', 'Ġriver', 'Ġrose', 'Ġabove', 'Ġits', 'Ġbanks', 'Ġthat', 'Ġnight']
XLM-RoBERTa (multi)       |   9 tokene | ratio: 1.12 | ['▁The', '▁river', '▁rose', '▁above', '▁its', '▁bank', 's', '▁that', '▁night']

Text: 'The flooding waters destroyed the ancient walls'
Nr cuvinte: 7
--------------------------------------------------
GPT-2 (BPE)               |   7 tokene | ratio: 1.0 | ['The', 'Ġflooding', 'Ġwaters', 'Ġdestroyed', 'Ġthe', 'Ġancient', 'Ġwalls']
BERT (WordPiece)          |   7 tokene | ratio: 1.0 | ['the', 'flooding', 'waters', 'destroyed', 'the', 'ancient', 'walls

## Cerinta 1b - fine tuning

In [6]:
split = ds.train_test_split(test_size=0.1)
tokenized = split.map(tokenize, batched=True, remove_columns=["line", "gutenberg_id"])
model = GPT2LMHeadModel.from_pretrained("gpt2")
training_args = TrainingArguments(
    output_dir="./gpt2-poetry",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="no",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained("./gpt2-poetry-finetuned")
tokenizer.save_pretrained("./gpt2-poetry-finetuned")


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,4.208253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-poetry-finetuned/tokenizer_config.json',
 './gpt2-poetry-finetuned/tokenizer.json')

In [9]:
tokenizer_ft = GPT2Tokenizer.from_pretrained("./gpt2-poetry-finetuned")
generator_ft = pipeline("text-generation", model="./gpt2-poetry-finetuned", tokenizer=tokenizer_ft, device=-1)
results_ft = []

for params in params_to_test:
    result = generator_ft(first_verse, generation_config=GenerationConfig(
        max_new_tokens=40,
        do_sample=True,
        pad_token_id=50256,
        **params
    ))
    text = result[0]['generated_text'].replace(first_verse, "").strip()
    results_ft.append({
        "temperature": params["temperature"],
        "top_p": params["top_p"],
        "rep_penalty": params["repetition_penalty"],
        "diversitate": round(lexical_diversity(text), 3),
        "repetitii": len(count_repetitions(text)),
        "lung_medie_cuv": round(avg_word_length(text), 2),
        "nr_tokene": token_count(text, tokenizer_ft)
    })

df_ft = pd.DataFrame(results_ft)
print(df_ft.to_string(index=False))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

 temperature  top_p  rep_penalty  diversitate  repetitii  lung_medie_cuv  nr_tokene
         0.3    0.9          1.0        0.235          2            4.35         38
         0.7    0.9          1.0        0.667          2            4.50         38
         1.2    0.9          1.0        0.727          2            4.27         37
         0.8    0.3          1.0        0.235          4            3.59         38
         0.8    0.5          1.0        0.353          2            4.76         38
         0.8    0.7          1.0        0.421          3            3.16         36
         0.8    0.9          1.5        1.000          0            5.17         10
         0.8    0.9          2.0        0.964          1            4.39         40


In [10]:
results_compare = []

for params in params_to_test:
    # Original
    result_orig = generator(first_verse, generation_config=GenerationConfig(
        max_new_tokens=40, do_sample=True, pad_token_id=50256, **params
    ))
    text_orig = result_orig[0]['generated_text'].replace(first_verse, "").strip()

    # Fine-tunat
    result_ft = generator_ft(first_verse, generation_config=GenerationConfig(
        max_new_tokens=40, do_sample=True, pad_token_id=50256, **params
    ))
    text_ft = result_ft[0]['generated_text'].replace(first_verse, "").strip()

    results_compare.append({
        "temperature": params["temperature"],
        "top_p": params["top_p"],
        "rep_penalty": params["repetition_penalty"],
        "text_original": text_orig,
        "text_finetuned": text_ft,
    })

# ============================================================
# AFISARE COMPARATIVA
# ============================================================
for r in results_compare:
    print("=" * 60)
    print(f"temp={r['temperature']} | top_p={r['top_p']} | rep_penalty={r['rep_penalty']}")
    print(f"\n[ORIGINAL]\n{r['text_original']}")
    print(f"\n[FINE-TUNAT]\n{r['text_finetuned']}")
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


temp=0.3 | top_p=0.9 | rep_penalty=1.0

[ORIGINAL]
and the sun shone down upon the sky. The sun was shining down upon the sky, and the moon was shining down upon the sky. The sun was shining down upon the sky, and the moon

[FINE-TUNAT]
And the river-gulls,

And the sea-gulls,

And the sea-gulls,

And the sea-gulls,

temp=0.7 | top_p=0.9 | rep_penalty=1.0

[ORIGINAL]
and was the place where the sun rose and the moon set. A deep and deep river ran through the sky, and the moon came to life. It was a beautiful river, and the moon was

[FINE-TUNAT]
And its banks rose up

And its banks

Saw the land of the wild-goose,

Thou art the wild-goose,

Thou

temp=1.2 | top_p=0.9 | rep_penalty=1.0

[ORIGINAL]
but the sound faded away as the moon sank farther into the horizon. A sudden and rapid wind, driven by sudden hunger and a terrible cold, rushed to shore with a sudden roar, and when all

[FINE-TUNAT]
Till it became in gloom with its roar;

Shrewd at him with wild fury;

Hearken his voice like

## Cerinta 1.c

#### 1.c1 - Care sunt diferentele de calitate intre textele generate cu cele doua tipuri de LLM-uri?
- diferenta este ca primul tip de LLM, nu respecta deloc structura unei poezii, nu este la fel de creativ, cuvinte mai less-poetic, repetitii mai multe, diversitate mai mica

#### 1.c2 Ce se intampla daca versurile din prompt sunt in limba engleza?

- Ambele modele functioneaza in engleza, deci nimic special nu se intampla, deoarece asa este intended sa fie folosit.

#### 1.c3 Ce se intampla daca versiurile din prompt sunt in limba romana?

- Depinde, in cazul meu, am raspuns la 1.c4, dar daca este intended sa fie in limba romana, atunci tokenizarea va decurge perfect si generarea va fi in regula

#### 1.c4 Ce se intampla daca versurile din prompt sunt in limba romana si corpusul de antrenare este in limba engleza?

- Aici apare problema, modelul continua sa genereze in engleza desi este limba romana, pentru ca el nu intelege limba romana, tokenele vor fi foarte ciudate si fara sens.

#### 1.c5 Cum se poate "personaliza" LLM pentru a genera versuri in stil de pastel (cu accent pe frumusetea naturii)?

- cu fine-tuning pe un dataset in stil de pastel cu accent pe frumusetea naturii

In [11]:
first_verse_ro = "Dunărea a crescut peste maluri în noaptea aceea,"

result_orig = generator(first_verse_ro, generation_config=GenerationConfig(
    max_new_tokens=40, temperature=0.8, top_p=0.9,
    repetition_penalty=1.5, do_sample=True, pad_token_id=50256
))

result_ft = generator_ft(first_verse_ro, generation_config=GenerationConfig(
    max_new_tokens=40, temperature=0.8, top_p=0.9,
    repetition_penalty=1.5, do_sample=True, pad_token_id=50256
))

print("[ORIGINAL - prompt romana]")
print(result_orig[0]['generated_text'])
print("\n[FINE-TUNAT - prompt romana]")
print(result_ft[0]['generated_text'])

# Analiza tokenizare
tokens_ro = tokenizer.encode(first_verse_ro)
tokens_en = tokenizer.encode("The river rose above its banks that night,")
print(f"\nTokene romana: {len(tokens_ro)}")
print(f"Tokene engleza: {len(tokens_en)}")
print(f"Tokene romana (detaliat): {tokenizer.convert_ids_to_tokens(tokens_ro)}")

[ORIGINAL - prompt romana]
Dunărea a crescut peste maluri în noaptea aceea, makráte rēo fotora e-aœur.


The same name is also given to the two places of power in Europe (see Parnassus

[FINE-TUNAT - prompt romana]
Dunărea a crescut peste maluri în noaptea aceea,
"The omen of the war."

 (Beneath this word) "O Ojibway!" said he. He looked up at her; and with his long-suffering

Tokene romana: 21
Tokene engleza: 9
Tokene romana (detaliat): ['Dun', 'Ä', 'ĥ', 'rea', 'Ġa', 'Ġc', 'res', 'cut', 'Ġpest', 'e', 'Ġmal', 'uri', 'ĠÃ', '®', 'n', 'Ġno', 'apt', 'ea', 'Ġace', 'ea', ',']
